# 09 – Apache Flink

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

Start: `docker compose --profile flink up -d flink` (in the folder `.devcontainer`).

Flink 2.3 runs as its own container (JobManager + TaskManager + **SQL Gateway**). PyFlink does not fit
the lab's Python environment (it pins old pyarrow/pandas versions), so we speak Flink SQL through the
SQL Gateway's REST API (`http://flink:8083`). The adapter `FlinkEngine` opens a gateway session and
registers the catalog's warehouses as Iceberg REST catalogs:

- `sts` → warehouse with vended or own credentials (Lakekeeper: `demo-sts`)
- `signing` → Lakekeeper only: `demo-signing` (remote signing – Flink uses the same Iceberg Java code as Spark for it)

The session runs in **batch mode** (`execution.runtime-mode=batch`, `table.dml-sync=true`): an `INSERT` only
returns once the job has committed.

In [ ]:
import pandas as pd

from iceberg_lab.config import WAREHOUSE_STS
from iceberg_lab.engines import FlinkEngine

flink = FlinkEngine(WAREHOUSE_STS)


def sql(q):
    """Run Flink SQL; SELECT results as DataFrame (column names are not part of the JSON rows)."""
    rows = flink.execute(q)
    return (
        pd.DataFrame(rows)
        if rows and q.lstrip().upper().startswith(("SELECT", "SHOW", "DESCRIBE"))
        else rows
    )


sql("SHOW CATALOGS")

## Create a table and write

The format (v2/v3) and Iceberg properties go into the `WITH` clause.

In [ ]:
sql("CREATE DATABASE IF NOT EXISTS sts.nb_flink")
import uuid

from iceberg_lab.notebook import catalog_limit


def fresh(name):
    """Drop the table of an earlier run; where the catalog cannot drop (RustFS refuses
    Flink's purge), continue under a new name."""
    dropped = False
    with catalog_limit(f"DROP TABLE {name}"):
        sql(f"DROP TABLE IF EXISTS sts.nb_flink.{name}")
        dropped = True
    return name if dropped else f"{name}_{uuid.uuid4().hex[:6]}"


ORDERS_N = fresh("orders")
ORDERS = f"sts.nb_flink.{ORDERS_N}"

created = False
with catalog_limit("v3 table"):
    sql(f"""CREATE TABLE {ORDERS} (order_id BIGINT, status STRING, amount DOUBLE)
           WITH ('format-version' = '3')""")
    created = True
if not created:  # e.g. RustFS: no v3 - continue with v2
    sql(f"""CREATE TABLE {ORDERS} (order_id BIGINT, status STRING, amount DOUBLE)
           WITH ('format-version' = '2')""")
sql(f"""INSERT INTO {ORDERS} VALUES
       (1, 'open', 10.0E0), (2, 'paid', 20.0E0), (3, 'paid', 30.0E0), (4, 'shipped', 40.0E0)""")
sql(f"SELECT status, count(*), sum(amount) FROM {ORDERS} GROUP BY status")

## Upsert instead of MERGE (equality deletes)

Flink has no `MERGE`, `UPDATE` or `DELETE` for Iceberg (the Iceberg sink does not implement the row-level
interfaces). Flink's way is the **upsert**: a table with a primary key and `write.upsert.enabled`. Every row replaces
the previous row with the same key – written as an **equality delete file** plus a new data file.

In [ ]:
CUSTOMERS_N = fresh("customers")
CUSTOMERS = f"sts.nb_flink.{CUSTOMERS_N}"
with catalog_limit("Upsert with primary key"):
    sql(f"""CREATE TABLE {CUSTOMERS} (id INT NOT NULL, name STRING, PRIMARY KEY (id) NOT ENFORCED)
           WITH ('format-version' = '2', 'write.upsert.enabled' = 'true')""")
    sql(f"INSERT INTO {CUSTOMERS} VALUES (1, 'Ada'), (2, 'Bob')")
    sql(f"INSERT INTO {CUSTOMERS} VALUES (2, 'Bea'), (3, 'Cy')")  # 2 is replaced, 3 is new
    print(sql(f"SELECT * FROM {CUSTOMERS} ORDER BY id"))

The metadata table `$files` shows the equality deletes (`content = 2`). Metadata tables need backticks in Flink.

In [ ]:
with catalog_limit("Metadata table $files"):
    print(
        sql(
            f"SELECT content, count(*) FROM sts.nb_flink.`{CUSTOMERS_N}$files` GROUP BY content ORDER BY content"
        )
    )

Not every engine can apply equality deletes: DuckDB, Spark, Trino and pg_lake can, **PyIceberg (and thus
Polars/Daft) cannot** (`PyIceberg does not yet support equality deletes`). DuckDB reads the table correctly:

In [ ]:
with catalog_limit("Read with DuckDB"):
    from iceberg_lab.engines import DuckDBEngine

    duck = DuckDBEngine(WAREHOUSE_STS)
    print(duck.execute(f"SELECT id, name FROM {duck.fq('nb_flink', CUSTOMERS_N)} ORDER BY id"))

## Time travel and branches

Both through *dynamic table options* (`/*+ OPTIONS(...) */`).

In [ ]:
sql(f"INSERT INTO {ORDERS} VALUES (5, 'open', 50.0E0)")
with catalog_limit("Time travel through the metadata table $snapshots"):
    snaps = sql(
        f"SELECT snapshot_id FROM sts.nb_flink.`{ORDERS_N}$snapshots` ORDER BY committed_at"
    )
    first = int(snaps.iloc[0, 0])
    print(sql(f"SELECT count(*) FROM {ORDERS} /*+ OPTIONS('snapshot-id'='{first}') */"))

In [ ]:
# Writing to a branch creates it; main stays unchanged.
sql(f"INSERT INTO {ORDERS} /*+ OPTIONS('branch'='dev') */ VALUES (99, 'test', 0.0E0)")
pd.DataFrame(
    {
        "main": sql(f"SELECT count(*) FROM {ORDERS}").iloc[0, 0],
        "dev": sql(f"SELECT count(*) FROM {ORDERS} /*+ OPTIONS('branch'='dev') */").iloc[0, 0],
    },
    index=["rows"],
)

## What Flink SQL (Iceberg 1.12) cannot do

All points are recorded in the tests as known gaps with the exact error message:

| Missing | Error message / reason |
|---|---|
| `DELETE`, `UPDATE`, `MERGE` | the sink does not implement `SupportsRowLevelDelete/Update` |
| rollback, compaction, expire through SQL | `getProcedure is not implemented for class FlinkCatalog` (only the Java API `TableMaintenance`) |
| creating tags, partition evolution | no SQL syntax |
| `TIME` with fractions of a second | Flink truncates `TIME` to seconds on write, to milliseconds on read |
| `unknown`, `variant` | no writing; the gateway cannot return `variant` as JSON |
| v3 `initial-default` | Flink returns `NULL` instead of the default |

In [ ]:
try:
    sql(f"DELETE FROM {ORDERS} WHERE order_id = 1")
except RuntimeError as e:
    print(str(e)[:160])

In [ ]:
flink.close()